## PROJET FIN D'ÉTUDES - INGÉNIEUR IA
PHASE 2 : FEATURE ENGINEERING
Auteur : [Votre Nom]
Date   : Février 2026

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("PHASE 2 - FEATURE ENGINEERING")
print("=" * 60)

## CHARGEMENT DES DATASETS PROPRES (issus de la Phase 1)

In [ ]:
print("\n📂 Chargement des datasets propres...")

df = pd.read_csv("dataset_clean.csv", parse_dates=[
    "Date_Creation", "Date_Prise_En_Charge", "Date_Cloture",
    "Date_Entree", "Date_Fin_Contrat"
])
df_abs = pd.read_csv("absences_clean.csv", parse_dates=["Date_Debut", "Date_Fin"])

print(f"  ✅ dataset_clean   : {len(df)} lignes | {df.shape[1]} colonnes")
print(f"  ✅ absences_clean  : {len(df_abs)} lignes")

## BLOC A — MÉTRIQUES TEMPORELLES

In [ ]:
print("\n📅 BLOC A : Métriques temporelles...")

# A1 - Délai entre création et prise en charge (en jours)
# → Mesure le temps d'attente avant qu'un agent prenne le dossier
df["delai_prise_en_charge_j"] = (
    df["Date_Prise_En_Charge"] - df["Date_Creation"]
).dt.days
# Valeurs négatives ou aberrantes → NaN
df.loc[df["delai_prise_en_charge_j"] < 0, "delai_prise_en_charge_j"] = np.nan
df.loc[df["delai_prise_en_charge_j"] > 60, "delai_prise_en_charge_j"] = np.nan

# A2 - Durée de traitement réelle (en jours)
# → Variable CIBLE principale pour le modèle de prédiction
df["duree_traitement_reelle_j"] = (
    df["Date_Cloture"] - df["Date_Prise_En_Charge"]
).dt.days
df.loc[df["duree_traitement_reelle_j"] <= 0, "duree_traitement_reelle_j"] = np.nan
df.loc[df["duree_traitement_reelle_j"] > 90, "duree_traitement_reelle_j"] = np.nan

# A3 - Durée totale bout en bout (création → clôture)
df["duree_totale_j"] = (
    df["Date_Cloture"] - df["Date_Creation"]
).dt.days
df.loc[df["duree_totale_j"] <= 0, "duree_totale_j"] = np.nan

# A4 - Variables calendaires (saisonnalité)
df["jour_semaine"]   = df["Date_Creation"].dt.dayofweek      # 0=Lundi, 6=Dimanche
df["nom_jour"]       = df["Date_Creation"].dt.day_name()
df["mois"]           = df["Date_Creation"].dt.month
df["trimestre"]      = df["Date_Creation"].dt.quarter
df["semaine_annee"]  = df["Date_Creation"].dt.isocalendar().week.astype("Int64")
df["annee"]          = df["Date_Creation"].dt.year

# A5 - Semaine du mois (S1/S2/S3/S4)
df["semaine_du_mois"] = ((df["Date_Creation"].dt.day - 1) // 7) + 1

# A6 - Indicateurs de pics bancaires connus
df["est_fin_de_mois"] = (df["Date_Creation"].dt.day >= 25).astype(int)
df["est_debut_mois"]  = (df["Date_Creation"].dt.day <= 5).astype(int)
df["est_lundi"]       = (df["jour_semaine"] == 0).astype(int)
df["est_vendredi"]    = (df["jour_semaine"] == 4).astype(int)

# A7 - Fin de trimestre (pics réglementaires bancaires)
derniers_mois_trimestre = [3, 6, 9, 12]
df["est_fin_trimestre"] = df["mois"].isin(derniers_mois_trimestre).astype(int)

print(f"  ✅ Variables temporelles créées : delai_prise_en_charge_j, duree_traitement_reelle_j,")
print(f"     duree_totale_j, jour_semaine, mois, trimestre, semaine_du_mois,")
print(f"     est_fin_de_mois, est_debut_mois, est_lundi, est_vendredi, est_fin_trimestre")

## BLOC B — MÉTRIQUES DE PERFORMANCE PAR AGENT

In [ ]:
print("\n👤 BLOC B : Performance par agent...")

# B1 - Productivité journalière : volume traité / temps déclaré
# Éviter la division par zéro
df["productivite_dossiers_par_heure"] = np.where(
    (df["Temps_Passe_Declare_Min"] > 0) & (df["Volume_Dossiers"] > 0),
    df["Volume_Dossiers"] / (df["Temps_Passe_Declare_Min"] / 60),
    np.nan
)
# Valeurs aberrantes → NaN (> 25 dossiers/heure irréaliste pour ce type d'activité)
df.loc[df["productivite_dossiers_par_heure"] > 25, "productivite_dossiers_par_heure"] = np.nan

# B2 - Charge journalière par agent (nb tâches créées ce jour)
charge_jour = (
    df.groupby(["Matricule_Agent", "Date_Creation"])
    .size()
    .reset_index(name="charge_journaliere_nb_taches")
)
df = df.merge(charge_jour, on=["Matricule_Agent", "Date_Creation"], how="left")

# B3 - Moyenne de productivité par agent sur toute la période
# → Permet de détecter les agents sous/sur-performants
moy_prod_agent = (
    df.groupby("Matricule_Agent")["productivite_dossiers_par_heure"]
    .mean()
    .reset_index(name="moy_productivite_agent")
)
df = df.merge(moy_prod_agent, on="Matricule_Agent", how="left")

# B4 - Écart entre temps déclaré et durée réelle (en minutes)
# → Détecte les anomalies de déclaration
df["ecart_temps_declare_vs_reel"] = np.where(
    df["duree_traitement_reelle_j"].notna() & df["Temps_Passe_Declare_Min"].notna(),
    df["Temps_Passe_Declare_Min"] - (df["duree_traitement_reelle_j"] * 8 * 60),
    np.nan
)

# B5 - Ancienneté de l'agent au moment de la tâche (en mois)
# → Les nouveaux agents (intérimaires, CDD) sont moins rapides
df["anciennete_agent_mois"] = np.where(
    df["Date_Entree"].notna(),
    ((df["Date_Creation"] - df["Date_Entree"]).dt.days / 30).round(1),
    np.nan
)
df.loc[df["anciennete_agent_mois"] < 0, "anciennete_agent_mois"] = np.nan

print(f"  ✅ Variables agent créées : productivite_dossiers_par_heure, charge_journaliere_nb_taches,")
print(f"     moy_productivite_agent, ecart_temps_declare_vs_reel, anciennete_agent_mois")

## BLOC C — ETP DISPONIBLES PAR JOUR (métrique clé !)

In [ ]:
print("\n⚡ BLOC C : Calcul des ETP disponibles par jour...")

# C1 - Convertir le temps de travail en coefficient numérique
def convertir_temps_travail(val):
    if pd.isna(val):
        return 1.0  # On suppose temps plein si non renseigné
    val = str(val).strip().replace('%', '')
    try:
        v = float(val)
        return v / 100 if v > 1 else v  # "80" → 0.8, "0.8" → 0.8
    except:
        if 'plein' in str(val).lower():
            return 1.0
        if 'partiel' in str(val).lower():
            return 0.8
        return 1.0

df["coeff_temps_travail"] = df["Temps_Travail"].apply(convertir_temps_travail)

# C2 - L'agent est-il encore sous contrat à la date de la tâche ?
df["contrat_actif_a_date"] = np.where(
    df["Date_Fin_Contrat"].isna(),
    1,  # CDI ou pas de date de fin → toujours actif
    (df["Date_Fin_Contrat"] >= df["Date_Creation"]).astype(int)
)

# C3 - L'agent est-il absent ce jour ?
# Version vectorisée : on construit une table (Matricule, jour) -> 1
# en explosant chaque période d'absence en jours individuels,
# puis on merge sur (Matricule_Agent, Date_Creation).
lignes_absence = []
df_abs_valide = df_abs.dropna(subset=["Date_Debut", "Date_Fin", "Matricule"])
for _, row in df_abs_valide.iterrows():
    debut, fin = row["Date_Debut"], row["Date_Fin"]
    if fin < debut:
        continue  # période incohérente déjà signalée en cleaning
    nb_jours = (fin - debut).days + 1
    if nb_jours > 90:
        continue  # garde-fou
    jours = pd.date_range(debut, fin, freq="D")
    for j in jours:
        lignes_absence.append((row["Matricule"], j))

df_absence_jours = pd.DataFrame(lignes_absence, columns=["Matricule_Agent", "Date_Creation"])
df_absence_jours = df_absence_jours.drop_duplicates()
df_absence_jours["est_absent"] = 1

df = df.merge(df_absence_jours, on=["Matricule_Agent", "Date_Creation"], how="left")
df["est_absent"] = df["est_absent"].fillna(0).astype(int)

# C4 - ETP effectif de l'agent ce jour
# ETP = coeff_temps_travail × contrat_actif × (1 - absent)
df["etp_agent_jour"] = (
    df["coeff_temps_travail"] *
    df["contrat_actif_a_date"] *
    (1 - df["est_absent"])
)

# C5 - ETP total disponible par jour (global ET par Service)
# IMPORTANT : etp_agent_jour est répété sur chaque ligne de tâche d'un agent.
# Pour le total ETP du jour il faut une seule valeur par (agent, jour) ;
# on déduplique avant de sommer, sinon un agent avec 18 tâches/jour
# compterait pour 18 ETP au lieu de 1.
etp_agent_unique = df.drop_duplicates(subset=["Matricule_Agent", "Date_Creation"])[
    ["Matricule_Agent", "Date_Creation", "Service", "etp_agent_jour"]
]

# Global : utile pour les KPIs généraux de pilotage
etp_par_jour_global = (
    etp_agent_unique.groupby("Date_Creation")["etp_agent_jour"]
    .sum()
    .reset_index(name="etp_total_disponible_jour")
)
df = df.merge(etp_par_jour_global, on="Date_Creation", how="left")

# Par Service : nécessaire pour la segmentation par équipe métier
# (fonctionnalités Dash : surcharge par service, réallocation équitable)
# NB : un agent est rattaché à un seul Service (celui de son agence/équipe),
# donc pas de double comptage entre services pour un même agent.
etp_par_jour_service = (
    etp_agent_unique.groupby(["Date_Creation", "Service"])["etp_agent_jour"]
    .sum()
    .reset_index(name="etp_disponible_service_jour")
)
df = df.merge(etp_par_jour_service, on=["Date_Creation", "Service"], how="left")

print(f"  ✅ etp_agent_jour créé (ETP individuel par tâche)")
print(f"  ✅ etp_total_disponible_jour créé (global)")
print(f"  ✅ etp_disponible_service_jour créé (par Service)")
print(f"  → ETP moyen/jour (global) : {etp_par_jour_global['etp_total_disponible_jour'].mean():.1f}")
print(f"  → ETP min/jour (global)   : {etp_par_jour_global['etp_total_disponible_jour'].min():.1f}")
print(f"  → ETP max/jour (global)   : {etp_par_jour_global['etp_total_disponible_jour'].max():.1f}")

## BLOC D — VARIABLES MÉTIER (complexité & charge)

In [ ]:
print("\n🏦 BLOC D : Variables métier...")

# D1 - Ratio de complexité : temps passé / moyenne du service
# → Mesure si un dossier est plus complexe que la normale pour ce service
moy_temps_service = (
    df.groupby("Service")["Temps_Passe_Declare_Min"]
    .mean()
    .reset_index(name="moy_temps_service")
)
df = df.merge(moy_temps_service, on="Service", how="left")

df["ratio_complexite"] = np.where(
    df["moy_temps_service"] > 0,
    df["Temps_Passe_Declare_Min"] / df["moy_temps_service"],
    np.nan
)

# D2 - Encodage numérique de la complexité déclarée
mapping_complexite_num = {"Simple": 1, "Moyen": 2, "Complexe": 3, "Non renseigné": 2}
df["complexite_num"] = df["Complexite"].map(mapping_complexite_num).fillna(2)

# D3 - Impact du type de contrat sur la productivité
# (basé sur les coefficients réels définis lors de la génération)
mapping_productivite_contrat = {
    "CDI":          1.00,
    "Prestataire":  0.90,
    "CDD":          0.85,
    "Intérimaire":  0.75,
    "Temps partiel":0.80,
    "Alternant":    0.60,
    "Non renseigné":0.85
}
df["coeff_productivite_contrat"] = df["Type_Contrat"].map(
    mapping_productivite_contrat
).fillna(0.85)

# D4 - Volume de dossiers entrants par jour (global ET par Service)
volume_jour_global = (
    df.groupby("Date_Creation")["Volume_Dossiers"]
    .sum()
    .reset_index(name="volume_total_entrant_jour")
)
df = df.merge(volume_jour_global, on="Date_Creation", how="left")

volume_jour_service = (
    df.groupby(["Date_Creation", "Service"])["Volume_Dossiers"]
    .sum()
    .reset_index(name="volume_entrant_service_jour")
)
df = df.merge(volume_jour_service, on=["Date_Creation", "Service"], how="left")

# D5 - Charge par ETP (l'indicateur de pilotage principal !)
# Version globale
df["charge_par_etp"] = np.where(
    df["etp_total_disponible_jour"] > 0,
    df["volume_total_entrant_jour"] / df["etp_total_disponible_jour"],
    np.nan
)

# Version par Service (utilisée pour la surcharge/réallocation par service)
df["charge_par_etp_service"] = np.where(
    df["etp_disponible_service_jour"] > 0,
    df["volume_entrant_service_jour"] / df["etp_disponible_service_jour"],
    np.nan
)

# Calcul du seuil d'alerte (percentile 75) - global et par service
seuil_alerte = df["charge_par_etp"].quantile(0.75)
df["alerte_surcharge"] = (df["charge_par_etp"] > seuil_alerte).astype(int)

# Seuil par service : chaque service a sa propre dynamique de charge,
# un seuil unique global ne serait pas pertinent pour comparer/réallouer entre services
seuils_service = df.groupby("Service")["charge_par_etp_service"].quantile(0.75)
df["seuil_alerte_service"] = df["Service"].map(seuils_service)
df["alerte_surcharge_service"] = (
    df["charge_par_etp_service"] > df["seuil_alerte_service"]
).astype(int)

print(f"  ✅ ratio_complexite, complexite_num, coeff_productivite_contrat")
print(f"  ✅ volume_total_entrant_jour, charge_par_etp (global)")
print(f"  ✅ volume_entrant_service_jour, charge_par_etp_service, alerte_surcharge_service (par Service)")
print(f"  ✅ alerte_surcharge (seuil global = {seuil_alerte:.1f} dossiers/ETP)")
print(f"  → Seuils par service :")
for s, v in seuils_service.items():
    print(f"     - {s:<25} : {v:.1f} dossiers/ETP")

## BLOC E — DATASET AGRÉGÉ PAR JOUR (pour la prévision J+7/J+30)

In [ ]:
print("\n📊 BLOC E : Agrégation journalière pour prévision...")

# Ce dataset est celui qu'on donnera à XGBoost pour prévoir la charge future
df_journalier = df.groupby("Date_Creation").agg(
    nb_taches_jour            = ("ID_Tache",                    "count"),
    volume_entrant_jour       = ("Volume_Dossiers",             "sum"),
    etp_disponible            = ("etp_total_disponible_jour",   "first"),
    charge_par_etp            = ("charge_par_etp",              "first"),
    alerte_surcharge          = ("alerte_surcharge",            "first"),
    temps_moyen_traitement    = ("Temps_Passe_Declare_Min",     "mean"),
    duree_traitement_mediane  = ("duree_traitement_reelle_j",   "median"),
    taux_complexes            = ("complexite_num",              "mean"),
    jour_semaine              = ("jour_semaine",                "first"),
    mois                      = ("mois",                        "first"),
    trimestre                 = ("trimestre",                   "first"),
    est_fin_de_mois           = ("est_fin_de_mois",             "first"),
    est_fin_trimestre         = ("est_fin_trimestre",           "first"),
    est_lundi                 = ("est_lundi",                   "first"),
    est_vendredi              = ("est_vendredi",                "first"),
    semaine_du_mois           = ("semaine_du_mois",             "first"),
).reset_index()

df_journalier = df_journalier.rename(columns={"Date_Creation": "date"})
df_journalier = df_journalier.sort_values("date").reset_index(drop=True)

# Ajout de features de lag (valeurs des 7 jours précédents)
# → Permet au modèle d'apprendre les tendances récentes
for lag in [1, 7, 14, 30]:
    df_journalier[f"volume_lag_{lag}j"] = df_journalier["volume_entrant_jour"].shift(lag)
    df_journalier[f"etp_lag_{lag}j"]    = df_journalier["etp_disponible"].shift(lag)

# Moyenne mobile sur 7 jours (lisse les pics ponctuels)
df_journalier["volume_moy_7j"] = (
    df_journalier["volume_entrant_jour"].rolling(window=7, min_periods=1).mean()
)
df_journalier["charge_moy_7j"] = (
    df_journalier["charge_par_etp"].rolling(window=7, min_periods=1).mean()
)

print(f"  ✅ Dataset journalier (global) : {len(df_journalier)} jours | {df_journalier.shape[1]} colonnes")
print(f"  ✅ Features de lag créées : lag 1j, 7j, 14j, 30j")
print(f"  ✅ Moyennes mobiles 7j créées")

## BLOC E.2 — DATASET AGRÉGÉ PAR JOUR ET PAR SERVICE

In [ ]:
print("\n📊 BLOC E.2 : Agrégation journalière par Service...")

# Une ligne = un (date, service). Permet la prévision et la détection
# de surcharge par service (fonctionnalités Dash : surcharge, réallocation).
df_journalier_service = df.groupby(["Date_Creation", "Service"]).agg(
    nb_taches_jour            = ("ID_Tache",                    "count"),
    volume_entrant_jour       = ("Volume_Dossiers",             "sum"),
    etp_disponible            = ("etp_disponible_service_jour", "first"),
    charge_par_etp            = ("charge_par_etp_service",      "first"),
    alerte_surcharge          = ("alerte_surcharge_service",    "first"),
    temps_moyen_traitement    = ("Temps_Passe_Declare_Min",     "mean"),
    duree_traitement_mediane  = ("duree_traitement_reelle_j",   "median"),
    taux_complexes            = ("complexite_num",              "mean"),
    jour_semaine              = ("jour_semaine",                "first"),
    mois                      = ("mois",                        "first"),
    trimestre                 = ("trimestre",                   "first"),
    est_fin_de_mois           = ("est_fin_de_mois",             "first"),
    est_fin_trimestre         = ("est_fin_trimestre",           "first"),
    est_lundi                 = ("est_lundi",                   "first"),
    est_vendredi              = ("est_vendredi",                "first"),
    semaine_du_mois           = ("semaine_du_mois",             "first"),
).reset_index()

df_journalier_service = df_journalier_service.rename(columns={"Date_Creation": "date"})
df_journalier_service = df_journalier_service.sort_values(["Service", "date"]).reset_index(drop=True)

# Features de lag et moyennes mobiles, calculées séparément pour chaque service
# (groupby + shift/rolling pour ne pas mélanger les séries entre services)
for lag in [1, 7, 14, 30]:
    df_journalier_service[f"volume_lag_{lag}j"] = (
        df_journalier_service.groupby("Service")["volume_entrant_jour"].shift(lag)
    )
    df_journalier_service[f"etp_lag_{lag}j"] = (
        df_journalier_service.groupby("Service")["etp_disponible"].shift(lag)
    )

df_journalier_service["volume_moy_7j"] = (
    df_journalier_service.groupby("Service")["volume_entrant_jour"]
    .rolling(window=7, min_periods=1).mean()
    .reset_index(drop=True)
)
df_journalier_service["charge_moy_7j"] = (
    df_journalier_service.groupby("Service")["charge_par_etp"]
    .rolling(window=7, min_periods=1).mean()
    .reset_index(drop=True)
)

print(f"  ✅ Dataset journalier par service : {len(df_journalier_service)} lignes "
      f"({df_journalier_service['Service'].nunique()} services x ~{df_journalier['date'].nunique()} jours)")
print(f"  ✅ Features de lag et moyennes mobiles calculées par service (sans fuite inter-service)")

## EXPORT FINAL

In [ ]:
print("\n💾 Export des datasets enrichis...")

df.to_csv("dataset_features.csv", index=False, encoding="utf-8")
df_journalier.to_csv("dataset_journalier.csv", index=False, encoding="utf-8")
df_journalier_service.to_csv("dataset_journalier_service.csv", index=False, encoding="utf-8")

print(f"  ✅ dataset_features.csv          → {len(df)} lignes | {df.shape[1]} colonnes")
print(f"  ✅ dataset_journalier.csv         → {len(df_journalier)} jours")
print(f"  ✅ dataset_journalier_service.csv → {len(df_journalier_service)} lignes (jours x services)")

## RAPPORT FINAL

In [ ]:
print("\n" + "=" * 60)
print("RAPPORT FINAL - FEATURE ENGINEERING")
print("=" * 60)

nouvelles_cols = [
    "delai_prise_en_charge_j", "duree_traitement_reelle_j", "duree_totale_j",
    "jour_semaine", "mois", "trimestre", "semaine_du_mois",
    "est_fin_de_mois", "est_debut_mois", "est_lundi", "est_vendredi", "est_fin_trimestre",
    "productivite_dossiers_par_heure", "charge_journaliere_nb_taches",
    "moy_productivite_agent", "ecart_temps_declare_vs_reel", "anciennete_agent_mois",
    "coeff_temps_travail", "contrat_actif_a_date", "est_absent",
    "etp_agent_jour", "etp_total_disponible_jour", "etp_disponible_service_jour",
    "ratio_complexite", "complexite_num", "coeff_productivite_contrat",
    "volume_total_entrant_jour", "volume_entrant_service_jour",
    "charge_par_etp", "charge_par_etp_service",
    "alerte_surcharge", "alerte_surcharge_service"
]

print(f"\n  Nouvelles features créées : {len(nouvelles_cols)}")
for i, col in enumerate(nouvelles_cols, 1):
    print(f"    {i:02d}. {col}")

print(f"\n  Statistiques clés :")
print(f"  → Durée traitement moyenne   : {df['duree_traitement_reelle_j'].mean():.1f} jours")
print(f"  → ETP moyen disponible/jour  : {df['etp_total_disponible_jour'].mean():.1f}")
print(f"  → Charge moyenne par ETP     : {df['charge_par_etp'].mean():.1f} dossiers/ETP")
print(f"  → Jours en alerte surcharge (global)  : {df_journalier['alerte_surcharge'].sum()} / {len(df_journalier)}")
print(f"  → (date,service) en alerte surcharge  : {df_journalier_service['alerte_surcharge'].sum()} / {len(df_journalier_service)}")
print(f"  → Taux absences détectées    : {df['est_absent'].mean()*100:.1f}%")

print("\n✅ Phase 2 terminée ! Prêt pour la modélisation (Phase 3).")
print("   Fichier global  -> prévision globale     : dataset_journalier.csv")
print("   Fichier service -> prévision par service : dataset_journalier_service.csv")
print("   Fichier détaillé -> Isolation Forest, dérive individuelle : dataset_features.csv")